In [1]:
import pandas as pd
import numpy as np

In [2]:
file_path = r'C:\dev\projects\NLP_test\data\train_dataset_manual.csv'
data = pd.read_csv(file_path)
data.describe()

,max(page),target,available
count,150,54,150
unique,150,52,2
top,https://www.factorybuys.com.au/products/euro-t...,Gift Card,True
freq,1,3,78


In [3]:
data['target'] = data['target'].fillna('NO_PRODUCT')

In [4]:
data.head(10)

,max(page),target,available
0,https://www.factorybuys.com.au/products/euro-t...,Factory Buys 32cm Euro Top Mattress - King,True
1,https://dunlin.com.au/products/beadlight-cirrus,Beadlight Cirrus LED Reading Light,True
2,https://themodern.net.au/products/hamar-plant-...,Hamar Plant Stand - Ash,True
3,https://furniturefetish.com.au/products/oslo-o...,NO_PRODUCT,False
4,https://hemisphereliving.com.au/products/,NO_PRODUCT,False
5,https://home-buy.com.au/products/bridger-penda...,NO_PRODUCT,False
6,https://interiorsonline.com.au/products/interi...,"$50 RJ Living Gift Card, $100 RJ Living Gift C...",True
7,https://beckurbanfurniture.com.au/products/pag...,NO_PRODUCT,False
8,https://livingedge.com.au/products/tables/dining,"Linear Wood Table, Saarinen Dining Table, Oval...",True
9,https://edenliving.online/collections/summerlo...,NO_PRODUCT,False


In [5]:
train_data = data[data["available"] == True]

In [6]:
train_data.describe()

,max(page),target,available
count,78,78,78
unique,78,52,1
top,https://www.factorybuys.com.au/products/euro-t...,NO_PRODUCT,True
freq,1,25,78


In [7]:
from sklearn.model_selection import train_test_split
X, y = train_data['max(page)'], train_data['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [8]:
import re
import json
from urllib.parse import urljoin, urlparse
from typing import Optional

import requests
from bs4 import BeautifulSoup, Tag

TIMEOUT = 30

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

_SKIP_TAGS = {"header", "footer", "aside"}
_SKIP_ATTRS = re.compile(
    r"\b(nav|footer|header|menu|breadcrumb|sidebar|filter|facet|pagination|"
    r"cart|checkout|newsletter|popup|modal|cookie|banner|subscribe|"
    r"social|share|wishlist-modal|refinement)\b",
    re.IGNORECASE,
)

def _clean(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

def _is_price(text: str) -> bool:
    return bool(re.match(r"^[\$€£¥₹]?\s?\d[\d,.\s]*$", text.strip()))

def _is_noise(text: str) -> bool:
    return False

def _in_skip_zone(tag: Tag) -> bool:
    for parent in tag.parents:
        if not isinstance(parent, Tag):
            continue
        if parent.name in _SKIP_TAGS:
            return True
        cls = " ".join(parent.get("class", []))
        id_ = parent.get("id", "")
        role = parent.get("role", "")
        check = f"{cls} {id_} {role}"
        if _SKIP_ATTRS.search(check):
            return True
    return False

def _fetch(url: str) -> tuple[Optional[BeautifulSoup], str, int]:
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        if response.status_code == 200:
            return BeautifulSoup(response.text, 'html.parser'), response.url, response.status_code
        return None, response.url, response.status_code
        
    except Exception as e:
        return None, url, 0 

def _was_redirected_away(original_url: str, final_url: Optional[str]) -> bool:
    if final_url is None:
        return True
    orig = urlparse(original_url)
    final = urlparse(final_url)
    if orig.netloc.lower() != final.netloc.lower():
        return True
    if orig.path.rstrip("/") and final.path.rstrip("/") == "":
        return True
    return False

def _from_jsonld(soup: BeautifulSoup) -> list[str]:
    products: list[str] = []
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string or "")
        except (json.JSONDecodeError, TypeError):
            continue
        _walk_ld(data, products)
    return products

def _walk_ld(node, out: list[str]):
    if isinstance(node, list):
        for item in node:
            _walk_ld(item, out)
        return
    if not isinstance(node, dict):
        return
    types = node.get("@type", "")
    types = types if isinstance(types, list) else [types]
    if any(t in ("Product", "IndividualProduct") for t in types):
        name = _clean(node.get("name", ""))
        if name:
            out.append(name)
        return
    if "ItemList" in types:
        for elem in node.get("itemListElement", []):
            _walk_ld(elem, out)
    if "@graph" in node:
        _walk_ld(node["@graph"], out)
    for key, val in node.items():
        if key.startswith("@"):
            continue
        if isinstance(val, (dict, list)):
            _walk_ld(val, out)

_PRODUCT_URL_PATTERNS = [
    re.compile(r"/[A-Z0-9][\w-]*\.html$", re.IGNORECASE),
    re.compile(r"/products/[\w-]+$", re.IGNORECASE),
    re.compile(r"/product/[\w-]+/?$", re.IGNORECASE),
    re.compile(r"/(?:p|item|dp)/[\w-]+/?$", re.IGNORECASE),
]

def _looks_like_product_url(href: str, base_host: str) -> bool:
    parsed = urlparse(href)
    if parsed.netloc and parsed.netloc != base_host:
        return False
    path = parsed.path
    for pat in _PRODUCT_URL_PATTERNS:
        if pat.search(path):
            return True
    return False


def _name_from_img_link(a_tag: Tag) -> Optional[str]:
    img = a_tag.find("img")
    if not img:
        return None
    for attr in ("title", "alt"):
        val = _clean(img.get(attr, "") or "")
        if val and not _is_noise(val) and not _is_price(val) and len(val) > 2:
            return val
    return None


def _name_from_text_link(a_tag: Tag) -> Optional[str]:
    lines = [_clean(l) for l in a_tag.get_text("\n").split("\n") if _clean(l)]
    for line in lines:
        if not _is_noise(line) and not _is_price(line) and len(line) > 2:
            return line
    return None

def _from_product_links(soup: BeautifulSoup, base_url: str) -> list[str]:
    base_host = urlparse(base_url).netloc
    found: dict[str, str] = {}

    for a in soup.find_all("a", href=True):
        href = urljoin(base_url, a["href"])
        norm = urlparse(href)._replace(query="", fragment="").geturl()

        if not _looks_like_product_url(href, base_host):
            continue
        if _in_skip_zone(a):
            continue
        if norm in found:
            continue

        name = _name_from_img_link(a) or _name_from_text_link(a)
        if name:
            found[norm] = name

    return list(found.values())

_CARD_PATTERN = re.compile(
    r"\b(product[-_]?card|product[-_]?tile|product[-_]?item|"
    r"grid[-_]?item|collection[-_]?item|plp[-_]item|"
    r"ProductCard|productCard)\b",
    re.IGNORECASE,
)


def _name_from_card(card: Tag) -> Optional[str]:
    for level in ("h2", "h3", "h4", "h1", "h5"):
        h = card.find(level)
        if h:
            txt = _clean(h.get_text())
            if txt and not _is_noise(txt) and not _is_price(txt):
                return txt
    for el in card.find_all(True):
        cls = " ".join(el.get("class", []))
        if re.search(r"(product|item|card)[_-]?(name|title)", cls, re.I):
            txt = _clean(el.get_text())
            if txt and not _is_noise(txt) and not _is_price(txt):
                return txt
    a = card.find("a")
    if a:
        return _name_from_text_link(a)
    return None

def _from_html_cards(soup: BeautifulSoup) -> list[str]:
    products: list[str] = []
    seen: set[str] = set()

    candidates = [
        tag for tag in soup.find_all(True)
        if _CARD_PATTERN.search(" ".join(tag.get("class", [])))
    ]
    filtered: list[Tag] = []
    for c in candidates:
        if not any(c in f.descendants for f in filtered):
            filtered = [f for f in filtered if f not in c.descendants]
            filtered.append(c)

    for card in filtered:
        if _in_skip_zone(card):
            continue
        name = _name_from_card(card)
        if name and name.lower() not in seen:
            seen.add(name.lower())
            products.append(name)

    return products
    
def _from_title(soup: BeautifulSoup) -> Optional[str]:
    title_tag = soup.find("title")
    if title_tag:
        raw = _clean(title_tag.get_text())
        txt = re.split(r"\s*[|–—]\s*", raw)[0].strip()
        txt = re.sub(r"\s*[-–]\s*(Shop|Store|Buy|Online).*$", "", txt, flags=re.I).strip()
        if txt and not _is_noise(txt):
            return txt
    return None

def scrape_some_page_info(url: str) -> list[str]:
    soup, final_url, status_code = _fetch(url)
    all_data = []

    all_data.append(f"STATUS_{status_code}")

    if not soup:
        return all_data

    all_data.extend(_from_jsonld(soup))
    
    all_data.extend(_from_product_links(soup, url))
    
    all_data.extend(_from_html_cards(soup))
    
    title = _from_title(soup)
    if title:
        all_data.append(title)

    all_data.append(_clean(soup.get_text()[:2000])) 

    seen = set()
    return [x for x in all_data if x and not (x in seen or seen.add(x))]

In [9]:
def _name_from_url(url: str) -> Optional[str]:
    path = urlparse(url).path.rstrip("/")
    if not path:
        return None
        
    segments = [s for s in path.split("/") if s]
    if not segments:
        return None

    slug = segments[-1]

    slug = re.sub(r"\.\w{2,4}$", "", slug)

    if re.match(r"^[A-Z0-9]+-[A-Z0-9]+$", slug) and len(segments) >= 2:
        slug = segments[-2]
        slug = re.sub(r"\.\w{2,4}$", "", slug)

    name = re.sub(r"[-_]+", " ", slug)
    name = name.strip().title()

    if not name or len(name) <= 2:
        return None

    return name

In [10]:
X_train_texts = [scrape_some_page_info(url) for url in X_train]
X_train_texts = [str(text) if text is not None else "" for text in X_train_texts]

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased") 

inputs = tokenizer(X_train_texts, padding = True, truncation = True, 
                   max_length = 512, return_offsets_mapping = True, return_tensors = "pt")

In [ ]:
#tag sample products tokens next